# STAC AI Workshop · NYC · 21 October 2026
## 1 · Fixed RAG versus an agent: which component does what?
**Matthew Dixon, PhD, FRM · BeliefLens**

Run each cell in order. Every input and output appears in a readable box.

**Our question:** Does Harbor Metals have comfortable covenant headroom?

| Workflow | Who retrieves documents? | Who calls the calculator? |
|---|---|---|
| Fixed RAG | Application code | Nobody |
| Fixed RAG + calculator | Application code | Application code |
| Agent | Model requests; application executes | Model requests; application executes |

**RAG means retrieval augmented generation:** send a query to a document search service, receive matching text, then include that text with the question sent to the LLM.

By the end, you will be able to locate the retrieval query, specify which documents can be searched, call the calculator, and inspect an agent's choices.

The default is a scripted synthetic LLM. Its implementation is in `architecture_support.py`; it illustrates the interface and does not measure real model ability.

### Open in your browser
[Open this notebook in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01_architectures.ipynb), or choose **File → Open notebook → GitHub** in Colab and paste its GitHub address. Choose **File → Save a copy in Drive** for your own copy.

Local JupyterLab users: keep `architecture_support.py` beside this notebook. Colab users: the setup below downloads that helper from this public repository. The helper contains the scripted model and provider adapter; the workflow itself stays visible below.

In [ ]:
# Colab companion-file setup. No model call and no API key required.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    from urllib.request import urlretrieve
    if not Path("architecture_support.py").exists():
        urlretrieve("https://raw.githubusercontent.com/mfrdixon/STAC_AI_Bootcamp/main/notebooks/architecture_support.py", "architecture_support.py")

In [1]:
from architecture_support import (
    DOCUMENTS, INITIAL_DOCUMENTS, QUESTION, DocumentSearch,
    ModelSession, calculate_leverage, financial_inputs, restricted_cash, panel,
)

# These are the documents available to this exercise.
rag = DocumentSearch(DOCUMENTS)
panel("Available documents", "\n\n".join(
    f"{id}: {doc['text']}" for id, doc in DOCUMENTS.items()
))
panel("Question sent to the model", QUESTION)

# Default: no key or network needed.
PROVIDER = "synthetic"  # optional: "openai" or "together"
MODEL = ""              # enter a tool-capable model ID for live mode

def new_model():
    return ModelSession(provider=PROVIDER, model=MODEL)


### A · Fixed RAG: retrieve → send documents → read answer
The **application** specifies both the query and the documents to search. Here it searches the filing and covenant. The annex is available but is outside this initial search.

`documents` is the retrieval output. `add_context` sends that returned text alongside the question to the model. No calculator is called.

**Before running:** find `query` and `document_ids` below. Those are the application's retrieval instructions.

**After running:** look for FILING and COVENANT in the returned passages. The application puts those passages in the model input. The model does not choose another search in this workflow.

In [2]:
query = "Harbor Metals debt cash EBITDA covenant"
documents = rag.search(query=query, document_ids=INITIAL_DOCUMENTS)
panel("RAG output · text returned for the query", documents)

model = new_model()
model.add_context(documents)
answer = model.next()
panel("Model output · fixed RAG", answer)


### B · Fixed RAG + calculator: retrieve → calculate → send both → read answer
The application extracts figures from the filing, supplies the calculator arguments, and sends its result to the model. `financial_inputs` is a small parser for this controlled document format.

**Inspect the cash input:** this simple pipeline uses all $15m of reported cash. The covenant refers to an annex. Does a correct calculation establish the right covenant ratio?

**Before running:** locate the actual function call `calculate_leverage(**inputs)`. The application runs it before asking for the answer.

**After running:** look for cash **15** and leverage **3.00×**. This arithmetic uses reported cash; it has not yet resolved the annex's restriction.

In [3]:
documents = rag.search(query=query, document_ids=INITIAL_DOCUMENTS)
inputs = financial_inputs(documents)
panel("Calculator input · chosen by application code", inputs)

calculation = calculate_leverage(**inputs)
panel("Calculator output", calculation)

model = new_model()
model.add_context(documents, calculation)
answer = model.next()
panel("Model output · fixed RAG + calculator", answer)


### C · Agent: ask model → execute its requested tool → return result → repeat
Here the model initially receives the question and two tool definitions: `retrieve` and `calculate_leverage`. It chooses a retrieval query, document IDs and calculator arguments. Python executes each request and sends the result back.

The loop below is the workflow. Follow the numbered boxes: **model request → tool result → next model request**. The six-turn limit bounds the investigation.

**Before running:** notice that the loop does not specify “search the annex next.” It asks the model for its next request.

**After running:** follow the boxes in order: initial retrieval → annex request → calculator with eligible cash **10** → answer. Python executes the requests. The default model is scripted to illustrate this sequence; live models may behave differently.

In [4]:
model = new_model()
agent_calculation = None
for step in range(1, 7):
    response = model.next(allow_tools=True)
    panel(f"Step {step} · model output", response)
    if not response.calls:
        break
    for call in response.calls:
        if call.name == "retrieve":
            result = rag.search(**call.arguments)
        elif call.name == "calculate_leverage":
            result = calculate_leverage(**call.arguments)
            agent_calculation = result
        else:
            raise ValueError("Tool not permitted")
        panel(f"Step {step} · {call.name} result", result)
        model.observe(call, result)
else:
    panel("Stop", "Tool budget reached; review required.")


# An answer is not proof that the model used the calculator.
if agent_calculation is None:
    panel("Verification", "NOT VERIFIED: no calculator call completed. Review the answer.")
else:
    panel("Executed calculator result", agent_calculation)
    panel("Limit of this check", "Confirms tool execution, not that every sentence agrees with the result.")

### Try one change: give the fixed workflow an annex step
An agent is not required when you can specify the needed steps in code. Here the application explicitly retrieves Annex A and excludes restricted cash before calculating.

Compare the returned ratio with the earlier **3.00×**. The result should be **3.25×**. A current calculation still leaves scenario risk and approval unresolved.

**Why this matters:** the agent has no exclusive ability to find the annex. A fixed workflow can explicitly retrieve it too. The comparison concerns where the next-step choice is made, not proof that an agent is more accurate.

In [5]:
annex = rag.search(query="restricted cash", document_ids=["ANNEX_A"])
panel("Extra RAG query and returned annex", annex)

if not annex["documents"]:
    panel("Decision", "HOLD: eligible cash cannot be established without the annex.")
else:
    inputs = financial_inputs(documents)
    inputs["eligible_cash"] -= restricted_cash(annex)  # read the returned annex
    panel("Corrected calculator input", inputs)
    panel("Corrected calculator output", calculate_leverage(**inputs))


### Class exercise
1. In the first retrieval cell, change `document_ids=INITIAL_DOCUMENTS` to `document_ids=["FILING"]`. Run that cell again. Which source disappears from the model's input?
2. Restore it. Before running the agent cell, run `rag.documents.pop("ANNEX_A", None)` in a new cell. Does the agent request the annex? What does retrieval return?
3. Restore the store with `rag = DocumentSearch(DOCUMENTS)`. Run the agent again and inspect the calculator's cash argument.

**What to notice:** retrieval supplies specific text; a calculator checks specific arguments; an agent chooses requests. Inspecting those hand-offs shows exactly what each answer is supported by.

### Optional live calls
Set `PROVIDER="openai"` or `"together"` in the setup cell and set `MODEL` to a tool-capable model available in your account. Re-run the setup and the workflow you want to try. Each new session requests your provider key through a hidden prompt. The kernel sends the synthetic documents directly to the selected provider; calls are billed to your account. Restart the kernel to clear keys.

Live models may choose different queries, skip steps or hit a limit. The boxes show what actually happened. The synthetic implementation and HTTP adapter are available in the helper file for optional inspection.

[OpenAI tool calling](https://developers.openai.com/api/docs/guides/function-calling) · [Together tool calling](https://docs.together.ai/docs/function-calling)

**Observed live behavior:** tested Together models sometimes retrieved evidence but skipped the calculator. One supplied an incorrect formula and ratio. The verification box flags the missing calculation; do not treat a fluent answer as evidence of tool use. The offline scripted path is an illustration, not a model-performance result.

## Continue: Harbor Metals tools, MCP and permissions
Open [01b_harbor_tools_mcp_permissions.ipynb](01b_harbor_tools_mcp_permissions.ipynb).

Try direct function calls first. Then see how the same tools can be exposed through MCP and how research, reviewer and revoked permissions change allowed actions. The core exercise runs without an API key. Optional live calls use OpenAI.

**Browser alternative:** in Google Colab choose **File → Open notebook → GitHub**, paste the notebook GitHub URL, then **Save a copy in Drive**. The companion notebook includes setup instructions. 